# Load , Clean Dataset

In [ ]:
import pandas as pd

path = r"C:\Users\Siva\OneDrive\Desktop\feedbackanalysissystem\data\Education_Feedback.csv"

df = pd.read_csv(path)

# Drop useless column
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Keep only required columns
df = df[["StudentComments", "Sentiment"]]

# Remove null rows
df = df.dropna()

print(df.shape)
print(df.head())


(2007747, 2)
                                     StudentComments Sentiment
0                                               good  positive
1                                               good  positive
2                                            teacher  positive
3  friendly teacher but not enough ability to enc...  positive
4                                            teacher  positive


# Basic Text Cleaning

In [3]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df["clean_text"] = df["StudentComments"].apply(clean_text)


# Rule-Based Word Dictionaries

In [4]:
positive_words = [
    "good","excellent","nice","friendly","helpful","amazing",
    "great","supportive","clear","awesome","love","best"
]

negative_words = [
    "bad","poor","worst","rude","confusing","boring",
    "slow","unclear","hate","terrible","problem"
]


# Rule-Based Sentiment Function

In [5]:
def rule_sentiment(text):

    words = text.split()

    pos = sum(word in positive_words for word in words)
    neg = sum(word in negative_words for word in words)

    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    else:
        return "neutral"


# Apply Rule Engine

In [6]:
df["rule_prediction"] = df["clean_text"].apply(rule_sentiment)

print(df[["StudentComments","Sentiment","rule_prediction"]].head())


                                     StudentComments Sentiment rule_prediction
0                                               good  positive        positive
1                                               good  positive        positive
2                                            teacher  positive         neutral
3  friendly teacher but not enough ability to enc...  positive        positive
4                                            teacher  positive         neutral


# Evaluate Model

In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = df["Sentiment"]
y_pred = df["rule_prediction"]

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))


Accuracy: 0.7325330332955298
              precision    recall  f1-score   support

    negative       0.55      0.16      0.25    141203
     neutral       0.11      0.31      0.16    144505
    positive       0.90      0.82      0.86   1722039

    accuracy                           0.73   2007747
   macro avg       0.52      0.43      0.42   2007747
weighted avg       0.82      0.73      0.76   2007747

[[  22775   58110   60318]
 [   6816   44184   93505]
 [  12053  306204 1403782]]


# Saving Rule Model

In [8]:
import json

rule_model = {
    "positive_words": positive_words,
    "negative_words": negative_words
}

with open("rule_model.json", "w") as f:
    json.dump(rule_model, f)

print("Rule Model Saved")


Rule Model Saved


# Predict New Feedback (example checking)

In [10]:
def predict_feedback(text):

    text = clean_text(text)
    return rule_sentiment(text)

print(predict_feedback("teacher is very friendly and helpful"))


positive
